# 第17章 债券投资策略与回测 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch17_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch17_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：骑乘曲线归因（图17-1）


In [ ]:
import numpy as np
from fi import backtest as bt, data, risk, plotting
from fi.cashflow import make_cashflows
plotting.use_chinese_style()
cv = data.load_sample('cgb_yield_curve'); tenor, yld = cv['tenor'].to_numpy(), (cv['yield_pct']/100).to_numpy()
buy = [2,3,5,7,10]; carries, rolls = [], []
for b in buy:
    yb=float(np.interp(b,tenor,yld)); ys=float(np.interp(b-1,tenor,yld))
    cf,t=make_cashflows(yb,b,1,100); d=risk.modified_duration(cf,t,yb,1)
    r=bt.riding_attribution(yb,ys,d,yb,1); carries.append(r['carry']*100); rolls.append(r['rolldown']*100)
    print(f'买{b}y: carry={r["carry"]*100:.2f}% roll-down={r["rolldown"]*100:.3f}% 总={r["total"]*100:.2f}%')
fig, ax = plotting.new_axes(); x=[f'{b}Y' for b in buy]
ax.bar(x, carries, label='carry'); ax.bar(x, rolls, bottom=carries, label='roll-down')
ax.set_xlabel('买入期限(持有1年)'); ax.set_ylabel('收益贡献 (%)'); ax.set_title('骑乘曲线归因'); ax.legend(); fig.tight_layout()


## 编程实验 7：久期择时回测 vs 买入持有（绩效对比）


In [ ]:
rng = np.random.default_rng(42)
passive = 0.025/252 + rng.normal(0,0.0006,252)
active = 0.030/252 + rng.normal(0,0.0015,252)
for name, r in [('被动(buy-hold)', passive), ('主动(久期择时)', active)]:
    p = bt.performance(r, 252)
    print(f'{name}: 年化={p["ann_return"]*100:.2f}% 波动={p["ann_vol"]*100:.2f}% 夏普={p["sharpe"]:.2f} 回撤={p["max_drawdown"]*100:.2f}%')
fig, ax = plotting.new_axes()
ax.plot(bt.nav(passive), label='被动'); ax.plot(bt.nav(active), label='主动')
ax.set_xlabel('交易日'); ax.set_ylabel('累计净值'); ax.set_title('主动vs被动'); ax.legend(); fig.tight_layout()
print('注意：主动收益更高但夏普可能更低(波动更大)')


## 编程实验 8：完整收益归因（carry/roll-down/久期/凸性）


In [ ]:
# 持有 5Y 国债一年，期间市场利率 +30bp
yb = float(np.interp(5,tenor,yld)); ys = float(np.interp(4,tenor,yld))
cf,t = make_cashflows(yb,5,1,100); d = risk.modified_duration(cf,t,yb,1); c = risk.convexity(cf,t,yb,1)
dy_mkt = 0.0030
carry = yb; rolldown = -d*(ys-yb); dur = -d*dy_mkt; convx = 0.5*c*dy_mkt**2
total = carry+rolldown+dur+convx
print(f'carry      = {carry*100:+.3f}%  (躺赢)')
print(f'roll-down  = {rolldown*100:+.3f}%  (躺赢)')
print(f'久期(利率) = {dur*100:+.3f}%  (主动alpha)')
print(f'凸性       = {convx*100:+.3f}%')
print(f'合计       = {total*100:+.3f}%')
